# 01 - 数据下载

本 Notebook 负责从 akshare 接口下载所有原始数据，包括：
1. 10 只自选 A 股的后复权日度行情
2. 沪深 300 和上证指数的日度数据
3. CPI 同比增速和 M2 同比增速的月度宏观数据
4. 10 只股票最近 5 个年度的 ROE 和净利润率

所有下载结果均记录至 `download_log.txt`。

In [1]:
# ========== 1. 导入库与配置 ==========
import os
import sys
import time
import warnings
from datetime import datetime, timedelta

import pandas as pd
import numpy as np
import akshare as ak
import baostock as bs

warnings.filterwarnings('ignore')

# 项目根目录
PROJ_DIR = os.path.abspath(os.getcwd())
DATA_DIR = os.path.join(PROJ_DIR, 'data')

# 股票信息：代码 -> (名称, 行业)
STOCKS = {
    '002142': ('宁波银行', '银行'),
    '600036': ('招商银行', '银行'),
    '002594': ('比亚迪', '汽车'),
    '600048': ('保利发展', '房地产'),
    '000063': ('中兴通讯', '通讯'),
    '600487': ('亨通光电', '通讯'),
    '600519': ('贵州茅台', '白酒'),
    '300750': ('宁德时代', '能源'),
    '600900': ('长江电力', '能源'),
    '002352': ('顺丰控股', '物流'),
}

# 指数代码（baostock 格式）
INDEX_CODES = {
    'sh.000300': '沪深300',
    'sh.000001': '上证指数',
}

# 日期范围
START_DATE = '2020-01-01'
END_DATE = datetime.now().strftime('%Y-%m-%d')
START_DATE_COMPACT = '20200101'
END_DATE_COMPACT = datetime.now().strftime('%Y%m%d')

print(f'项目目录: {PROJ_DIR}')
print(f'数据时间范围: {START_DATE} ~ {END_DATE}')
print(f'股票数量: {len(STOCKS)}')
print(f'指数数量: {len(INDEX_CODES)}')

项目目录: e:\lianyujun_homework\my own\dshw-p01
数据时间范围: 2020-01-01 ~ 2026-05-23
股票数量: 10
指数数量: 2


In [2]:
# ========== 2. 创建目录结构 ==========
dirs = [
    os.path.join(DATA_DIR, 'stock'),
    os.path.join(DATA_DIR, 'index'),
    os.path.join(DATA_DIR, 'macro'),
    os.path.join(DATA_DIR, 'finance'),
    os.path.join(DATA_DIR, 'clean'),
    os.path.join(DATA_DIR, 'combined'),
    os.path.join(PROJ_DIR, 'output'),
]

for d in dirs:
    os.makedirs(d, exist_ok=True)
    print(f'已创建/确认: {d}')

已创建/确认: e:\lianyujun_homework\my own\dshw-p01\data\stock
已创建/确认: e:\lianyujun_homework\my own\dshw-p01\data\index
已创建/确认: e:\lianyujun_homework\my own\dshw-p01\data\macro
已创建/确认: e:\lianyujun_homework\my own\dshw-p01\data\finance
已创建/确认: e:\lianyujun_homework\my own\dshw-p01\data\clean
已创建/确认: e:\lianyujun_homework\my own\dshw-p01\data\combined
已创建/确认: e:\lianyujun_homework\my own\dshw-p01\output


In [3]:
# ========== 3. 下载日志辅助函数 ==========
LOG_FILE = os.path.join(PROJ_DIR, 'download_log.txt')

def write_log(status, name, shape_or_error):
    """记录下载日志，格式: [时间戳] SUCCESS/FAILED name shape=(行,列)"""
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    if status.upper() == 'SUCCESS':
        msg = f'[{timestamp}] SUCCESS  {name}  shape={shape_or_error}'
    else:
        msg = f'[{timestamp}] FAILED   {name}  Error: {shape_or_error}'
    with open(LOG_FILE, 'a', encoding='utf-8') as f:
        f.write(msg + '\n')
    print(msg)

# 清空旧日志
with open(LOG_FILE, 'w', encoding='utf-8') as f:
    f.write('# 数据下载日志\n')
print('日志文件已初始化')

日志文件已初始化


---
## 4. 下载个股后复权日度行情

使用 `akshare.stock_zh_a_hist()` 接口，设置 `adjust='hfq'`（后复权），下载每只股票的日度数据。

In [4]:
# ========== 4. 下载个股后复权日度行情（使用 baostock）==========
# baostock 的 adjustflag: 1=不复权, 2=后复权, 3=前复权

# 登录 baostock
lg = bs.login()
print(f'baostock 登录: {lg.error_code} {lg.error_msg}')

# 转换股票代码为 baostock 格式（SH/SZ 前缀）
def to_bs_code(code):
    if code.startswith('6') or code.startswith('9'):
        return f'sh.{code}'
    else:
        return f'sz.{code}'

for code, (name, industry) in STOCKS.items():
    try:
        bs_code = to_bs_code(code)
        rs = bs.query_history_k_data_plus(
            bs_code,
            fields='date,open,close,high,low,volume,amount',
            start_date=START_DATE,
            end_date=END_DATE,
            frequency='d',
            adjustflag='2'  # 后复权
        )
        
        if rs.error_code != '0':
            write_log('FAILED', f'stock_{code}', f'baostock error: {rs.error_msg}')
            continue
        
        # 转换为 DataFrame
        df = rs.get_data()
        if df is None or df.empty:
            write_log('FAILED', f'stock_{code}', 'No data returned')
            continue
        
        # 转换数据类型
        for col in ['open', 'close', 'high', 'low', 'volume', 'amount']:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        
        # 保存 CSV
        save_path = os.path.join(DATA_DIR, 'stock', f'stock_{code}.csv')
        df.to_csv(save_path, index=False, encoding='utf-8-sig')
        
        write_log('SUCCESS', f'stock_{code}', df.shape)
        time.sleep(0.3)
        
    except Exception as e:
        write_log('FAILED', f'stock_{code}', str(e))

bs.logout()
print('\n个股行情下载完成！')

login success!
baostock 登录: 0 success
[2026-05-23 22:57:09] SUCCESS  stock_002142  shape=(1545, 7)
[2026-05-23 22:57:15] SUCCESS  stock_600036  shape=(1545, 7)
[2026-05-23 22:57:19] SUCCESS  stock_002594  shape=(1545, 7)
[2026-05-23 22:57:22] SUCCESS  stock_600048  shape=(1545, 7)
[2026-05-23 22:57:23] SUCCESS  stock_000063  shape=(1545, 7)
[2026-05-23 22:57:27] SUCCESS  stock_600487  shape=(1545, 7)
[2026-05-23 22:57:29] SUCCESS  stock_600519  shape=(1545, 7)
[2026-05-23 22:57:35] SUCCESS  stock_300750  shape=(1545, 7)
[2026-05-23 22:57:37] SUCCESS  stock_600900  shape=(1545, 7)
[2026-05-23 22:57:41] SUCCESS  stock_002352  shape=(1545, 7)
logout success!

个股行情下载完成！


---
## 5. 下载市场指数数据

下载沪深 300（000300）作为 CAPM 市场基准，以及上证指数（000001）作为额外参考。

In [5]:
# ========== 5. 下载指数数据（使用 baostock）==========
lg = bs.login()
print(f'baostock 登录: {lg.error_code} {lg.error_msg}')

for bs_code, idx_name in INDEX_CODES.items():
    try:
        rs = bs.query_history_k_data_plus(
            bs_code,
            fields='date,open,close,high,low,volume',
            start_date=START_DATE,
            end_date=END_DATE,
            frequency='d',
            adjustflag='2'  # 后复权
        )
        
        if rs.error_code != '0':
            write_log('FAILED', f'index_{bs_code.split(".")[1]}', f'baostock error: {rs.error_msg}')
            continue
        
        df = rs.get_data()
        if df is None or df.empty:
            write_log('FAILED', f'index_{bs_code.split(".")[1]}', 'No data returned')
            continue
        
        # 转换数据类型
        for col in ['open', 'close', 'high', 'low', 'volume']:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        
        idx_code = bs_code.split('.')[1]
        save_path = os.path.join(DATA_DIR, 'index', f'index_{idx_code}.csv')
        df.to_csv(save_path, index=False, encoding='utf-8-sig')
        
        write_log('SUCCESS', f'index_{idx_code}', df.shape)
        time.sleep(0.3)
        
    except Exception as e:
        idx_code = bs_code.split('.')[1]
        write_log('FAILED', f'index_{idx_code}', str(e))

bs.logout()
print('\n指数数据下载完成！')

login success!
baostock 登录: 0 success
[2026-05-23 22:57:46] SUCCESS  index_000300  shape=(1545, 6)
[2026-05-23 22:57:48] SUCCESS  index_000001  shape=(1545, 6)
logout success!

指数数据下载完成！


---
## 6. 下载宏观经济指标

下载 CPI 同比增速（必选）和 M2 同比增速（自选）的月度数据。

**选择 M2 的理由**：M2 反映市场流动性，与股票市场的资金面密切相关，M2 增速变化往往影响市场走势。

In [6]:
# ========== 6. 下载宏观经济数据 ==========

# --- 6.1 CPI 同比增速（使用 macro_china_cpi，包含'全国-同比增长'列）---
try:
    df_cpi = ak.macro_china_cpi()
    if df_cpi is not None and not df_cpi.empty:
        df_cpi.columns = [str(c).strip() for c in df_cpi.columns]
        print('CPI 数据列名:', df_cpi.columns.tolist())
        # 保存，后续清洗时提取所需字段
        save_path = os.path.join(DATA_DIR, 'macro', 'macro_cpi.csv')
        df_cpi.to_csv(save_path, index=False, encoding='utf-8-sig')
        write_log('SUCCESS', 'macro_cpi', df_cpi.shape)
    else:
        write_log('FAILED', 'macro_cpi', 'No data returned')
except Exception as e:
    write_log('FAILED', 'macro_cpi', str(e))

time.sleep(0.5)

# --- 6.2 M2 同比增速（使用 macro_china_money_supply）---
try:
    df_m2 = ak.macro_china_money_supply()
    if df_m2 is not None and not df_m2.empty:
        df_m2.columns = [str(c).strip() for c in df_m2.columns]
        print('M2 数据列名:', df_m2.columns.tolist())
        save_path = os.path.join(DATA_DIR, 'macro', 'macro_m2.csv')
        df_m2.to_csv(save_path, index=False, encoding='utf-8-sig')
        write_log('SUCCESS', 'macro_m2', df_m2.shape)
    else:
        write_log('FAILED', 'macro_m2', 'No data returned')
except Exception as e:
    write_log('FAILED', 'macro_m2', str(e))

print('\n宏观数据下载完成！')

CPI 数据列名: ['月份', '全国-当月', '全国-同比增长', '全国-环比增长', '全国-累计', '城市-当月', '城市-同比增长', '城市-环比增长', '城市-累计', '农村-当月', '农村-同比增长', '农村-环比增长', '农村-累计']
[2026-05-23 22:57:52] SUCCESS  macro_cpi  shape=(220, 13)
M2 数据列名: ['月份', '货币和准货币(M2)-数量(亿元)', '货币和准货币(M2)-同比增长', '货币和准货币(M2)-环比增长', '货币(M1)-数量(亿元)', '货币(M1)-同比增长', '货币(M1)-环比增长', '流通中的现金(M0)-数量(亿元)', '流通中的现金(M0)-同比增长', '流通中的现金(M0)-环比增长']
[2026-05-23 22:57:52] SUCCESS  macro_m2  shape=(220, 10)

宏观数据下载完成！


---
## 7. 下载财务指标

获取 10 只股票最近 5 个年度的 ROE（净资产收益率）和净利润率，整理为长格式。

In [7]:
# ========== 7. 下载财务指标 ==========
# akshare stock_financial_abstract 返回格式：指标为行，日期为列
# 需要提取"净资产收益率(ROE)"和"销售净利率"两行

all_finance = []

for code, (name, industry) in STOCKS.items():
    try:
        # 获取财务数据
        df_fin = ak.stock_financial_abstract(symbol=code)
        
        if df_fin is None or df_fin.empty:
            write_log('FAILED', f'finance_{code}', 'No data returned')
            continue
        
        df_fin.columns = [str(c).strip() for c in df_fin.columns]
        print(f'{name}({code}) 财务指标行: {df_fin["指标"].tolist()}')
        
        # 查找 ROE 行和 销售净利率行
        roe_row = df_fin[df_fin['指标'].str.contains('净资产收益率', na=False)]
        profit_row = df_fin[df_fin['指标'].str.contains('销售净利率', na=False)]
        
        # 日期列（排除'选项'和'指标'列）
        date_cols = [c for c in df_fin.columns if c not in ['选项', '指标']]
        
        # 提取 ROE
        if len(roe_row) > 0:
            roe_vals = roe_row.iloc[0]
            for col in date_cols:
                val = pd.to_numeric(roe_vals[col], errors='coerce')
                if pd.notna(val) and len(str(col)) >= 6:
                    year = str(col)[:4]
                    current_year = datetime.now().year
                    if current_year - 5 <= int(year) < current_year:
                        all_finance.append({
                            'code': code, 'name': name, 'industry': industry,
                            'year': year, 'indicator': 'ROE', 'value': val
                        })
        
        # 提取 销售净利率
        if len(profit_row) > 0:
            profit_vals = profit_row.iloc[0]
            for col in date_cols:
                val = pd.to_numeric(profit_vals[col], errors='coerce')
                if pd.notna(val) and len(str(col)) >= 6:
                    year = str(col)[:4]
                    current_year = datetime.now().year
                    if current_year - 5 <= int(year) < current_year:
                        all_finance.append({
                            'code': code, 'name': name, 'industry': industry,
                            'year': year, 'indicator': '净利润率', 'value': val
                        })
        
        write_log('SUCCESS', f'finance_{code}', f'({len(df_fin)}, {len(df_fin.columns)})')
        time.sleep(0.3)
        
    except Exception as e:
        write_log('FAILED', f'finance_{code}', str(e))

# 整理为长格式并保存
if all_finance:
    df_finance = pd.DataFrame(all_finance)
    save_path = os.path.join(DATA_DIR, 'finance', 'finance_ratios.csv')
    df_finance.to_csv(save_path, index=False, encoding='utf-8-sig')
    print(f'\n财务数据已保存: {save_path}')
    print(df_finance.to_string())
else:
    print('\n警告: 未能获取到财务数据，请检查 akshare 接口或手动补充')

print('\n所有数据下载完成！')

宁波银行(002142) 财务指标行: ['归母净利润', '营业总收入', '营业成本', '净利润', '扣非净利润', '股东权益合计(净资产)', '商誉', '经营现金流量净额', '基本每股收益', '每股净资产', '每股现金流', '净资产收益率(ROE)', '总资产报酬率(ROA)', '毛利率', '销售净利率', '期间费用率', '资产负债率', '基本每股收益', '稀释每股收益', '摊薄每股收益_最新股数', '摊薄每股净资产_期末股数', '调整每股净资产_期末股数', '每股净资产_最新股数', '每股经营现金流', '每股现金流量净额', '每股企业自由现金流量', '每股股东自由现金流量', '每股未分配利润', '每股资本公积金', '每股盈余公积金', '每股留存收益', '每股营业收入', '每股营业总收入', '每股息税前利润', '净资产收益率(ROE)', '摊薄净资产收益率', '净资产收益率_平均', '净资产收益率_平均_扣除非经常损益', '摊薄净资产收益率_扣除非经常损益', '息税前利润率', '总资产报酬率', '总资本回报率', '投入资本回报率', '息前税后总资产报酬率_平均', '毛利率', '销售净利率', '成本费用利润率', '营业利润率', '总资产净利率_平均', '总资产净利率_平均(含少数股东损益)', '归母净利润', '营业总收入', '净利润', '扣非净利润', '营业总收入增长率', '归属母公司净利润增长率', '经营活动净现金/销售收入', '经营性现金净流量/营业总收入', '成本费用率', '期间费用率', '销售成本率', '经营活动净现金/归属母公司的净利润', '所得税/利润总额', '流动比率', '速动比率', '保守速动比率', '资产负债率', '权益乘数', '权益乘数(含少数股权的净资产)', '产权比率', '现金比率', '应收账款周转率', '应收账款周转天数', '存货周转率', '存货周转天数', '总资产周转率', '总资产周转天数', '流动资产周转率', '流动资产周转天数', '应付账款周转率']
[2026-05-23 22:57:56] SUCCESS  finance_002142  shape=(80, 87)
招商银

In [8]:
# ========== 8. 查看日志摘要 ==========
print('=== 下载日志摘要 ===')
with open(LOG_FILE, 'r', encoding='utf-8') as f:
    content = f.read()
print(content)

=== 下载日志摘要 ===
# 数据下载日志
[2026-05-23 22:57:09] SUCCESS  stock_002142  shape=(1545, 7)
[2026-05-23 22:57:15] SUCCESS  stock_600036  shape=(1545, 7)
[2026-05-23 22:57:19] SUCCESS  stock_002594  shape=(1545, 7)
[2026-05-23 22:57:22] SUCCESS  stock_600048  shape=(1545, 7)
[2026-05-23 22:57:23] SUCCESS  stock_000063  shape=(1545, 7)
[2026-05-23 22:57:27] SUCCESS  stock_600487  shape=(1545, 7)
[2026-05-23 22:57:29] SUCCESS  stock_600519  shape=(1545, 7)
[2026-05-23 22:57:35] SUCCESS  stock_300750  shape=(1545, 7)
[2026-05-23 22:57:37] SUCCESS  stock_600900  shape=(1545, 7)
[2026-05-23 22:57:41] SUCCESS  stock_002352  shape=(1545, 7)
[2026-05-23 22:57:46] SUCCESS  index_000300  shape=(1545, 6)
[2026-05-23 22:57:48] SUCCESS  index_000001  shape=(1545, 6)
[2026-05-23 22:57:52] SUCCESS  macro_cpi  shape=(220, 13)
[2026-05-23 22:57:52] SUCCESS  macro_m2  shape=(220, 10)
[2026-05-23 22:57:56] SUCCESS  finance_002142  shape=(80, 87)
[2026-05-23 22:57:57] SUCCESS  finance_600036  shape=(80, 103)
[202

In [9]:
# ========== 9. 验证所有文件 ==========
print('=== 已下载的文件 ===')
for root, dirs, files in os.walk(DATA_DIR):
    for f in files:
        fp = os.path.join(root, f)
        size_kb = os.path.getsize(fp) / 1024
        print(f'  {fp}  ({size_kb:.1f} KB)')

print(f'\n总计 {sum(len(files) for _, _, files in os.walk(DATA_DIR))} 个文件')

=== 已下载的文件 ===
  e:\lianyujun_homework\my own\dshw-p01\data\finance\finance_ratios.csv  (19.2 KB)
  e:\lianyujun_homework\my own\dshw-p01\data\index\index_000001.csv  (95.8 KB)
  e:\lianyujun_homework\my own\dshw-p01\data\index\index_000300.csv  (95.6 KB)
  e:\lianyujun_homework\my own\dshw-p01\data\macro\macro_cpi.csv  (20.7 KB)
  e:\lianyujun_homework\my own\dshw-p01\data\macro\macro_m2.csv  (21.4 KB)
  e:\lianyujun_homework\my own\dshw-p01\data\stock\stock_000063.csv  (117.0 KB)
  e:\lianyujun_homework\my own\dshw-p01\data\stock\stock_002142.csv  (122.9 KB)
  e:\lianyujun_homework\my own\dshw-p01\data\stock\stock_002352.csv  (121.3 KB)
  e:\lianyujun_homework\my own\dshw-p01\data\stock\stock_002594.csv  (117.8 KB)
  e:\lianyujun_homework\my own\dshw-p01\data\stock\stock_300750.csv  (125.6 KB)
  e:\lianyujun_homework\my own\dshw-p01\data\stock\stock_600036.csv  (120.1 KB)
  e:\lianyujun_homework\my own\dshw-p01\data\stock\stock_600048.csv  (117.1 KB)
  e:\lianyujun_homework\my own\ds